In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pymcts.games.bridgit.neural_net import BridgitNet
from pymcts.games.bridgit.config import BoardConfig, NeuralNetConfig
from pymcts.core.config import MCTSConfig
from pymcts.core.arena import Arena
from pymcts.core.players import MCTSPlayer
from pymcts.core.game_record import GameRecordCollection
from pymcts.games.bridgit.visualizer import Visualizer
from pymcts.games.bridgit.game import BridgitGame

## 1. Load training run

Set the `RUN_DIR` to point at the training run you want to analyse.

In [ ]:
RUN_DIR = Path("trainings/run_2026-03-20_113656")

# Load config
with open(RUN_DIR / "config.json") as f:
    run_config = json.load(f)

# Discover iterations
iter_dirs = sorted(RUN_DIR.glob("iteration_*"))
print(f"Run: {RUN_DIR.name}")
print(f"Board size: {run_config['board']['size']}")
print(f"Iterations found: {len(iter_dirs)}")

# Load all iteration data
iterations = []
for d in iter_dirs:
    data_path = d / "iteration_data.json"
    if data_path.exists():
        with open(data_path) as f:
            data = json.load(f)
        data["dir"] = d
        iterations.append(data)

print(f"Iterations with data: {len(iterations)}")
accepted = [i["iteration"] for i in iterations if i["arena"]["accepted"]]
print(f"Accepted iterations: {accepted}")

## 2. Training loss curves

Policy loss, value loss, and total loss across all iterations and epochs.

In [ ]:
# Collect per-epoch losses across all iterations
epoch_pi, epoch_v, epoch_total = [], [], []
epoch_x = []  # global epoch counter
x = 0
iter_boundaries = []  # x positions where a new iteration starts

for it in iterations:
    iter_boundaries.append(x)
    for ep in it["training"]["epochs"]:
        epoch_pi.append(ep["policy_loss"])
        epoch_v.append(ep["value_loss"])
        epoch_total.append(ep["total_loss"])
        epoch_x.append(x)
        x += 1

fig = make_subplots(rows=1, cols=2, subplot_titles=("All epochs (continuous)", "Final epoch per iteration"))

# Left: continuous epoch view
fig.add_trace(go.Scatter(x=epoch_x, y=epoch_pi, name="policy", line=dict(color="blue")), row=1, col=1)
fig.add_trace(go.Scatter(x=epoch_x, y=epoch_v, name="value", line=dict(color="orange")), row=1, col=1)
fig.add_trace(go.Scatter(x=epoch_x, y=epoch_total, name="total", line=dict(color="green")), row=1, col=1)

# Iteration boundary lines
for bx in iter_boundaries:
    fig.add_vline(x=bx, line_dash="dot", line_color="gray", opacity=0.3, row=1, col=1)

fig.update_xaxes(title_text="Global epoch", row=1, col=1)
fig.update_yaxes(title_text="Loss", row=1, col=1)

# Right: final loss per iteration
iter_nums = [it["iteration"] for it in iterations]
final_pi = [it["training"]["final_policy_loss"] for it in iterations]
final_v = [it["training"]["final_value_loss"] for it in iterations]
final_total = [it["training"]["final_total_loss"] for it in iterations]

fig.add_trace(go.Scatter(x=iter_nums, y=final_pi, name="policy (final)", line=dict(color="blue", dash="dash")), row=1, col=2)
fig.add_trace(go.Scatter(x=iter_nums, y=final_v, name="value (final)", line=dict(color="orange", dash="dash")), row=1, col=2)
fig.add_trace(go.Scatter(x=iter_nums, y=final_total, name="total (final)", line=dict(color="green", dash="dash")), row=1, col=2)

# Mark accepted iterations
for it in iterations:
    if it["arena"]["accepted"]:
        fig.add_vline(x=it["iteration"], line_dash="solid", line_color="green", opacity=0.2, row=1, col=2)

fig.update_xaxes(title_text="Iteration", row=1, col=2)
fig.update_yaxes(title_text="Loss", row=1, col=2)

fig.update_layout(height=400, width=1100, title_text="Training Losses")
fig.show()

## 3. Arena results

Win rate of the new model per iteration, and average game length for wins vs losses.

In [ ]:
iter_nums = [it["iteration"] for it in iterations]
win_rates = [it["arena"]["win_rate"] for it in iterations]
avg_win_moves = [it["arena"]["avg_moves_in_wins"] for it in iterations]
avg_loss_moves = [it["arena"]["avg_moves_in_losses"] for it in iterations]
accepted_flags = [it["arena"]["accepted"] for it in iterations]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Win rate (new vs prev)", "Avg game length"))

# Win rate
colors = ["green" if a else "red" for a in accepted_flags]
fig.add_trace(go.Bar(x=iter_nums, y=win_rates, marker_color=colors, name="win rate", showlegend=False), row=1, col=1)
fig.add_hline(y=run_config.get("arena", {}).get("threshold", 0.55), line_dash="dash", line_color="gray", row=1, col=1)
fig.update_yaxes(title_text="Win rate", range=[0, 1], row=1, col=1)
fig.update_xaxes(title_text="Iteration", row=1, col=1)

# Avg game length
fig.add_trace(go.Scatter(x=iter_nums, y=avg_win_moves, name="wins", line=dict(color="green")), row=1, col=2)
fig.add_trace(go.Scatter(x=iter_nums, y=avg_loss_moves, name="losses", line=dict(color="red")), row=1, col=2)
fig.update_yaxes(title_text="Avg moves", row=1, col=2)
fig.update_xaxes(title_text="Iteration", row=1, col=2)

fig.update_layout(height=400, width=1100, title_text="Arena Evaluation")
fig.show()

## 4. Browse games from an iteration

Pick an iteration and browse its self-play and eval games.

In [ ]:
ITERATION = 8  # <-- change this to browse a different iteration

iter_dir = RUN_DIR / f"iteration_{ITERATION:03d}"

# Load iteration data
with open(iter_dir / "iteration_data.json") as f:
    iter_data = json.load(f)

print(f"Iteration {ITERATION}")
print(f"  Training examples: {iter_data['training']['num_examples']}")
print(f"  Final loss: pi={iter_data['training']['final_policy_loss']:.4f}, "
      f"v={iter_data['training']['final_value_loss']:.4f}")
print(f"  Arena: new={iter_data['arena']['new_wins']}, prev={iter_data['arena']['prev_wins']} "
      f"(win rate: {iter_data['arena']['win_rate']:.1%})")
print(f"  Accepted: {iter_data['arena']['accepted']}")

# Load games
self_play_games = GameRecordCollection.model_validate_json(
    (iter_dir / "self_play_games.json").read_text()
)
eval_games = GameRecordCollection.model_validate_json(
    (iter_dir / "eval_games.json").read_text()
)
print(f"\n  Self-play games: {len(self_play_games)}")
print(f"  Eval games: {len(eval_games)}")

In [ ]:
# Self-play game distribution: moves per game
sp_moves = [g.num_moves for g in self_play_games]
fig = go.Figure(go.Histogram(x=sp_moves, nbinsx=20))
fig.update_layout(
    title=f"Iteration {ITERATION} — Self-play game lengths ({len(self_play_games)} games)",
    xaxis_title="Number of moves", yaxis_title="Count",
    height=350, width=700,
)
fig.show()

In [ ]:
# Browse a specific self-play game
GAME_INDEX = 0  # <-- change this

game = self_play_games[GAME_INDEX]
print(game.summary())
Visualizer.visualize_game(game)

In [ ]:
# Browse a specific eval game
EVAL_GAME_INDEX = 0  # <-- change this

eval_game = eval_games[EVAL_GAME_INDEX]
print(eval_game.summary())
Visualizer.visualize_game(eval_game)

## 5. Play with the trained models

Load pre-training and post-training checkpoints and pit them against each other.

In [ ]:
board_config = BoardConfig(size=run_config["board"]["size"])
mcts_config = MCTSConfig(num_simulations=200)

# Load post-training (best) and pre-training checkpoints for this iteration
post_net = BridgitNet(board_config)
post_net.load_checkpoint(str(iter_dir / "post_training.pt"))
pre_net = BridgitNet(board_config)
pre_net.load_checkpoint(str(iter_dir / "pre_training.pt"))

post_player = MCTSPlayer(net=post_net, mcts_config=mcts_config, name="post_training", temperature=1.0)
pre_player = MCTSPlayer(net=pre_net, mcts_config=mcts_config, name="pre_training", temperature=1.0)

arena = Arena(post_player, pre_player, game_factory=lambda: BridgitGame(board_config))

In [ ]:
# Play a single game and visualize
record = arena.play_game(verbose=True)
print(record.summary())
Visualizer.visualize_game(record)

In [ ]:
# Run a batch of games
collection = arena.play_games(num_games=20, verbose=True, swap_players=True)
print(f"\nResults: {collection.scores}")